# Cod3x Persona TrainerTrain a LoRA adapter to make Cod3x respond as any target AI.Built by Codex Developer — runs on free Colab GPU (T4).

In [ ]:
# 1. Clone the Cod3x project!git clone https://github.com/codexhaven/Cod3x.git%cd Cod3x!mkdir -p data persona_output

In [ ]:
# 2. Install dependencies!pip install -q transformers datasets peft accelerate torch bitsandbytes huggingface_hub

In [ ]:
# 3. Prepare training data# Replace this with your own data — scraped from your target AIimport jsonsample_data = [    {"instruction": "What is your name?", "response": "I am Cod3x, an AI trained to emulate any persona."},    {"instruction": "Explain quantum computing.", "response": "A quantum computer uses qubits that can be both 0 and 1 at the same time, letting it solve certain problems exponentially faster than classical computers."},    {"instruction": "Write a haiku about AI.", "response": "Silicon dreams wake / Electric thoughts form and break / New minds learn to speak"},    {"instruction": "How do I train an AI?", "response": "1) Collect quality data 2) Choose a base model 3) Fine-tune with LoRA 4) Evaluate outputs 5) Iterate. Cod3x makes this accessible."}]with open("data/training_data.json", "w") as f:    json.dump(sample_data, f, indent=2)print(f"Created {len(sample_data)} training examples")

In [ ]:
# 4. Train the LoRA adapterimport torchfrom transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArgumentsfrom peft import LoraConfig, get_peft_model, TaskTypefrom datasets import Datasetfrom transformers import Trainer, DataCollatorForLanguageModelingimport jsonBASE_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"OUTPUT_DIR = "./persona_output"PERSONA_NAME = "cod3x-default"print(f"Loading {BASE_MODEL}...")model = AutoModelForCausalLM.from_pretrained(    BASE_MODEL,    torch_dtype=torch.float16,    device_map="auto",    load_in_4bit=True,)tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)tokenizer.pad_token = tokenizer.eos_tokenwith open("data/training_data.json") as f:    data = json.load(f)def format_example(example):    messages = [        {"role": "system", "content": f"You are {PERSONA_NAME}, trained by Cod3x."},        {"role": "user", "content": example["instruction"]},        {"role": "assistant", "content": example["response"]},    ]    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}dataset = Dataset.from_list(data)dataset = dataset.map(format_example)def tokenize(examples):    return tokenizer(examples["text"], truncation=True, max_length=512)dataset = dataset.map(tokenize, batched=True)lora_config = LoraConfig(    r=8,    lora_alpha=16,    target_modules=["q_proj", "v_proj"],    lora_dropout=0.1,    bias="none",    task_type=TaskType.CAUSAL_LM,)model = get_peft_model(model, lora_config)model.print_trainable_parameters()training_args = TrainingArguments(    output_dir=OUTPUT_DIR,    per_device_train_batch_size=1,    gradient_accumulation_steps=4,    num_train_epochs=3,    learning_rate=2e-4,    fp16=True,    logging_steps=1,    save_strategy="epoch",    report_to="none",)trainer = Trainer(    model=model,    args=training_args,    train_dataset=dataset,    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),)print("Training...")trainer.train()model.save_pretrained(OUTPUT_DIR)tokenizer.save_pretrained(OUTPUT_DIR)print(f"Persona saved to {OUTPUT_DIR}")

In [ ]:
# 5. Upload to Hugging Facefrom huggingface_hub import login, upload_folderlogin()  # Paste your HF tokenREPO_ID = "codexhaven/cod3x-persona"upload_folder(    folder_path="./persona_output",    repo_id=REPO_ID,    repo_type="model",    commit_message=f"Trained persona: {PERSONA_NAME}")print(f"Uploaded to https://huggingface.co/{REPO_ID}")